# Routing agents with index-metadata Knowledge Indicators

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kderusso/elasticsearch-labs/blob/main/notebooks/context/manual-walkthrough/index-metadata-kis.ipynb)

This notebook is the runnable companion to Example 1 of the [Context management in Elasticsearch technical walkthrough](https://www.elastic.co/search-labs/blog/context-management-technical-walkthrough) blog.

Context is a retrieval problem, not a memory problem. A **Knowledge Indicator (KI)** is a pre-computed unit of context, stored in an **AI Index** and retrieved in a single call so an agent doesn't have to rediscover it on every query.

Here we tackle **routing**: given several Elasticsearch indices, help an agent pick the right one instead of guessing or searching all of them. We profile each index into an `index_metadata_entry` KI with a Kibana Workflow, then compare an agent answering the same question with and without those KIs.

## Prerequisites

This notebook is designed to run against an Elastic Serverless project, where the `ai-index-` component templates and Kibana Workflows are available. If you don't have one, [sign up for a trial](https://cloud.elastic.co/registration?onboarding_token=search&cta=cloud-registration&tech=trial&plcmt=article%20content&pg=search-labs).

You will need:

- Your Elasticsearch and Kibana endpoint URLs and an API key.
- A GenAI connector configured in Kibana (Stack Management → Connectors) for the workflow's `ai.agent` step. Serverless projects come pre-configured with the Elastic Inference Service and a default connector.
- An OpenAI-compatible API key to use with a deep agent harness. This notebook has been tested using OpenRouter API keys.

## Install packages and import modules

In [3]:
!pip install -q "elasticsearch>=9,<10" datasets requests langchain-openai langchain-core deepagents


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


### Initialize the Elasticsearch client

Connect with your **Elasticsearch** and **Kibana** endpoint URLs and an API key — the same inputs work for an Elastic Cloud deployment or a Serverless project.

- [Create an API key](https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key)

In [5]:
import json
import time
import requests
from getpass import getpass
from elasticsearch import Elasticsearch, helpers

ES_URL = input("Elasticsearch endpoint URL: ").strip().rstrip("/")
KIBANA_URL = input("Kibana endpoint URL: ").strip().rstrip("/")
ELASTIC_API_KEY = getpass("Elastic API key: ")

client = Elasticsearch(hosts=[ES_URL], api_key=ELASTIC_API_KEY)
print(client.info())

Elasticsearch endpoint URL:  http://localhost:9200
Kibana endpoint URL:  http://localhost:5601
Elastic API key:  ········


{'name': 'KathleenElastic', 'cluster_name': 'elasticsearch', 'cluster_uuid': 'oQrcAriOSTumpDcDsP2MKA', 'version': {'number': '9.6.0-SNAPSHOT', 'build_flavor': 'default', 'build_type': 'tar', 'build_hash': '1db7a4511be6c6d75180fef04bbe1ef720ad7352', 'build_date': '2026-07-24T02:04:24.844282044Z', 'build_snapshot': True, 'lucene_version': '10.5.0', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


Confirm Kibana is reachable — we drive the Workflows API through it.

In [6]:
def kbn_request(method, path, *, body=None, api_version=None):
    """Call a Kibana REST API and return the parsed JSON response."""
    headers = {
        "Authorization": f"ApiKey {ELASTIC_API_KEY}",
        "kbn-xsrf": "true",
        "Content-Type": "application/json",
    }
    if api_version:
        headers["elastic-api-version"] = api_version
    resp = requests.request(
        method,
        f"{KIBANA_URL}{path}",
        headers=headers,
        data=json.dumps(body) if body is not None else None,
    )
    resp.raise_for_status()
    return resp.json() if resp.text else {}


status = kbn_request("GET", "/api/status")
print("Kibana status:", status.get("status", {}).get("overall", {}).get("level"))

Kibana status: available


## Create some sample data

We start with three [BEIR benchmark](https://github.com/beir-cellar/beir) corpora from distinct domains, each in its own BM25-only index enriched with mapping metadata (`_meta.description` and per-field `meta.description`) — the human-written context the profiling workflow reads.

| Dataset | Domain | Index |
|---------|--------|-------|
| [FiQA](https://huggingface.co/datasets/BeIR/fiqa) | Financial Q&A | `beir-fiqa` |
| [NFCorpus](https://huggingface.co/datasets/BeIR/nfcorpus) | Biomedical / nutrition | `beir-nfcorpus` |
| [SciFact](https://huggingface.co/datasets/BeIR/scifact) | Scientific fact-checking | `beir-scifact` |

An agent with a question and these three indices has no idea which one is relevant at the start.

In [7]:
from datasets import load_dataset

SAMPLE_DOCS = 50

DATASETS = [
    {
        "hf_dataset": "BeIR/fiqa",
        "hf_config": "corpus",
        "hf_split": "corpus",
        "index_name": "beir-fiqa",
        "meta_description": (
            "FiQA: financial question answering corpus from StackExchange Finance "
            "community posts and web crawls. Covers investments, banking, taxes, "
            "and market analysis. BM25-only index."
        ),
    },
    {
        "hf_dataset": "BeIR/nfcorpus",
        "hf_config": "corpus",
        "hf_split": "corpus",
        "index_name": "beir-nfcorpus",
        "meta_description": (
            "NFCorpus: biomedical information retrieval corpus from NutritionFacts.org. "
            "Contains nutrition science and medical research documents on diet, disease, "
            "and health interventions. BM25-only index."
        ),
    },
    {
        "hf_dataset": "BeIR/scifact",
        "hf_config": "corpus",
        "hf_split": "corpus",
        "index_name": "beir-scifact",
        "meta_description": (
            "SciFact: scientific fact-checking corpus of biomedical research abstracts "
            "used to verify factual claims in peer-reviewed literature. BM25-only index."
        ),
    },
]

PROPERTIES = {
    "title": {"type": "text", "meta": {"description": "Document or article title."}},
    "text": {"type": "text", "meta": {"description": "Full document body text."}},
}


def beir_actions(ds, n):
    corpus = load_dataset(
        ds["hf_dataset"], ds["hf_config"], split=ds["hf_split"], streaming=True
    )
    for i, row in enumerate(corpus):
        if i >= n:
            break
        yield {
            "_index": ds["index_name"],
            "_id": row["_id"],
            "_source": {
                "title": row.get("title", "").strip()
                or " ".join(row.get("text", "").split())[:100],
                "text": row.get("text", ""),
            },
        }


for ds in DATASETS:
    name = ds["index_name"]
    client.indices.delete(index=name, ignore_unavailable=True)
    client.indices.create(
        index=name,
        mappings={
            "_meta": {"description": ds["meta_description"]},
            "properties": PROPERTIES,
        },
    )
    helpers.bulk(client, beir_actions(ds, SAMPLE_DOCS))
    client.indices.refresh(index=name)
    print(f"Indexed {client.count(index=name)['count']} documents into '{name}'.")

Indexed 50 documents into 'beir-fiqa'.
Indexed 50 documents into 'beir-nfcorpus'.
Indexed 50 documents into 'beir-scifact'.


## Create your AI Index

KIs live in an **AI Index**. The naming convention is what triggers automatic configuration: any index whose name starts with `ai-index-idx-` is a standard index; `ai-index-ds-` is a data stream. When Elasticsearch sees the prefix it applies component templates that configure the right mappings — `title`, `description`, and `content` each get a `.semantic` [`semantic_text`](https://www.elastic.co/docs/reference/elasticsearch/mapping-reference/semantic-text) sub-field for hybrid retrieval, alongside `type`, `tags`, `attributes`, and `references`.

In [8]:
AI_INDEX = "ai-index-idx-my-corpus"

# Recreate the AI Index. Passing no mappings lets the `ai-index-` component
# templates configure the standard KI fields (title/description/content with
# semantic_text sub-fields, plus type, tags, attributes, references).
client.indices.delete(index=AI_INDEX, ignore_unavailable=True)
client.indices.create(index=AI_INDEX)
print(f"Created AI Index: {AI_INDEX}\n")

print(json.dumps(client.indices.get_mapping(index=AI_INDEX).body, indent=2))

Created AI Index: ai-index-idx-my-corpus

{
  "ai-index-idx-my-corpus": {
    "mappings": {
      "properties": {
        "@timestamp": {
          "type": "date"
        },
        "attributes": {
          "type": "flattened"
        },
        "content": {
          "type": "text",
          "fields": {
            "semantic": {
              "type": "semantic_text",
              "inference_id": ".elser-2-elasticsearch"
            }
          }
        },
        "description": {
          "type": "text",
          "fields": {
            "semantic": {
              "type": "semantic_text",
              "inference_id": ".elser-2-elasticsearch"
            }
          }
        },
        "references": {
          "properties": {
            "uri": {
              "type": "keyword"
            }
          }
        },
        "tags": {
          "type": "keyword"
        },
        "title": {
          "type": "text",
          "fields": {
            "semantic": {
              "

## The query-ki skill

A KI is just a document in the AI Index, and finding one is a single ES|QL query. We package that query as a small, portable [Agent Skill](https://www.anthropic.com/news/skills) — a `SKILL.md` with a YAML header plus instructions — so any harness can load it. The query runs a hybrid (lexical + semantic) search, fuses the results with RRF, and filters by KI `type`.

We write it to disk so the deep agent below can load it from the `skills/` directory.

In [9]:
import os

SKILL_MD = """---
name: query-ki
description: >-
  Retrieve Knowledge Indicators (pre-computed context) from the Elasticsearch AI
  Index before answering. Use it to find which index to search (routing profiles)
  or to look up pre-computed facts without reading source documents. Trigger on any question that depends on specific facts, names, dates, or on choosing a data source.
allowed-tools: esql_query
---

# Retrieving Knowledge Indicators

Knowledge Indicators (KIs) live in Elasticsearch indices named `ai-index-*`.
Retrieve them by calling the `esql_query` tool with the query below. Substitute
the user's question for `<query>`, and choose the KI type you need: `corpus_entry`
for facts, `index_metadata_entry` for routing profiles.

```esql
FROM ai-index-idx-* METADATA _id, _index, _score
| WHERE type == "<ki_type>"
| FORK
    (WHERE MATCH(content, "<query>") OR MATCH(description, "<query>")
     | SORT _score DESC | LIMIT 20)
    (WHERE content.semantic : "<query>"
     | SORT _score DESC | LIMIT 20)
| FUSE
| SORT _score DESC
| KEEP title, content, description, tags
| LIMIT 5
```

Ground your answer in what the query returns, and cite the KI titles you used. If
nothing relevant comes back, say so rather than guessing.
"""

os.makedirs("skills/query-ki", exist_ok=True)
with open("skills/query-ki/SKILL.md", "w") as f:
    f.write(SKILL_MD)
print("Wrote skills/query-ki/SKILL.md")

Wrote skills/query-ki/SKILL.md


## Baseline: an agent without KIs

Both answers in this notebook come from the same [LangChain deep agents](https://pypi.org/project/deepagents/) harness — same model, same question — changing only how the agent retrieves context. First the baseline: the agent knows the three index names but not which is relevant, so it inspects mappings with `get_mapping` and searches with `esql_query`. Watch the tool-call count.

In [10]:
import threading

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import AIMessage
from langchain_core.callbacks import get_usage_metadata_callback
from deepagents import create_deep_agent
from deepagents.backends.filesystem import FilesystemBackend

# Any OpenAI-compatible endpoint works (OpenRouter, OpenAI, a local vLLM/Ollama, ...).
# Press Enter at the prompts to accept the defaults: OpenRouter + Claude Sonnet 4.5.
LLM_BASE_URL = (
    input("LLM base URL [https://openrouter.ai/api/v1]: ").strip()
    or "https://openrouter.ai/api/v1"
)
LLM_MODEL = (
    input("LLM model [anthropic/claude-sonnet-4.5]: ").strip()
    or "anthropic/claude-sonnet-4.5"
)
LLM_API_KEY = getpass("LLM API key: ")

# The deep agent plans and may chain several tool calls, so give it output headroom.
agent_llm = ChatOpenAI(
    base_url=LLM_BASE_URL,
    model=LLM_MODEL,
    api_key=LLM_API_KEY,
    max_tokens=4096,
)


@tool
def esql_query(query: str) -> list[dict] | str:
    """Execute an ES|QL query against Elasticsearch and return the matching rows.

    Args:
        query: A complete ES|QL query string, e.g. 'FROM beir-fiqa | LIMIT 5'.
               Full-text search syntax: WHERE MATCH(field, "value") -- not field MATCH "value".
    """
    try:
        resp = client.esql.query(query=query, format="json")
        cols = [c["name"] for c in resp["columns"]]
        return [dict(zip(cols, row)) for row in resp["values"]]
    except Exception as e:
        return f"ES|QL error: {e}"


@tool
def get_mapping(index: str) -> dict:
    """Return the field mapping for an Elasticsearch index or pattern."""
    return client.indices.get_mapping(index=index).body


def run_agent(agent, question):
    """Invoke the agent, print the tool calls it made and its answer.

    The headline metric is the number of tool calls -- with KIs the agent reaches
    the same answer in far fewer steps. Token usage is printed as a bonus cost signal.
    """
    print("Running agent", end="", flush=True)
    done = threading.Event()

    def _heartbeat():
        while not done.wait(5):
            print(".", end="", flush=True)

    hb = threading.Thread(target=_heartbeat, daemon=True)
    hb.start()
    try:
        with get_usage_metadata_callback() as cb:
            result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    finally:
        done.set()
        hb.join()
    print()

    print("\n--- Tool calls ---")
    for m in result["messages"]:
        if isinstance(m, AIMessage) and m.tool_calls:
            for tc in m.tool_calls:
                print(f"  [{tc['name']}] {str(tc['args'])[:120]}")
    total = sum(
        len(m.tool_calls)
        for m in result["messages"]
        if isinstance(m, AIMessage) and m.tool_calls
    )
    print(f"Total: {total}\n")

    usage = next(iter(cb.usage_metadata.values()), {})
    print(
        f"[tokens] input={usage.get('input_tokens', 0):,}  "
        f"output={usage.get('output_tokens', 0):,}  "
        f"total={usage.get('total_tokens', 0):,}\n"
    )

    print("--- Answer ---")
    print(result["messages"][-1].content)

LLM base URL [https://openrouter.ai/api/v1]:  
LLM model [anthropic/claude-sonnet-4.5]:  
LLM API key:  ········


In [11]:
QUESTION = (
    "Is there scientific evidence that vitamin D supplementation prevents cancer?"
)

baseline_agent = create_deep_agent(
    model=agent_llm,
    tools=[esql_query, get_mapping],
    system_prompt=(
        "You are a research assistant with access to three Elasticsearch indices: "
        "beir-fiqa, beir-nfcorpus, and beir-scifact. "
        "You do NOT know which index is relevant for a given question. "
        "Use get_mapping to inspect an index's description and fields, "
        "then query the most relevant one with esql_query. "
        'Full-text search syntax: WHERE MATCH(field, "value") -- never use field MATCH "value". '
        "Ground your answer strictly in what the queries return."
    ),
)

run_agent(baseline_agent, QUESTION)


--- Tool calls ---
  [get_mapping] {'index': 'beir-scifact'}
  [get_mapping] {'index': 'beir-nfcorpus'}
  [get_mapping] {'index': 'beir-fiqa'}
  [esql_query] {'query': 'FROM beir-scifact | WHERE MATCH(text, "vitamin D supplementation cancer prevention") OR MATCH(text, "vitamin 
  [esql_query] {'query': 'FROM beir-nfcorpus | WHERE MATCH(text, "vitamin D supplementation cancer prevention") OR MATCH(text, "vitamin
  [esql_query] {'query': 'FROM beir-scifact | WHERE MATCH(text, "vitamin D cancer trial randomized") | LIMIT 10'}
  [esql_query] {'query': 'FROM beir-nfcorpus | WHERE MATCH(text, "vitamin D cancer randomized trial supplementation") | LIMIT 10'}
  [esql_query] {'query': 'FROM beir-nfcorpus | WHERE MATCH(text, "vitamin D cancer incidence mortality outcome") | LIMIT 15'}
  [esql_query] {'query': 'FROM beir-scifact | WHERE MATCH(text, "vitamin D supplementation") | LIMIT 15'}
  [esql_query] {'query': 'FROM beir-nfcorpus | WHERE MATCH(title, "vitamin D cancer") OR MATCH(title, "vita

{'messages': [HumanMessage(content='Is there scientific evidence that vitamin D supplementation prevents cancer?', additional_kwargs={}, response_metadata={}, id='53eea150-8c61-4400-9565-f258498ca740'),
  AIMessage(content="I'll search the scientific databases to find evidence about vitamin D supplementation and cancer prevention.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 161, 'prompt_tokens': 7229, 'total_tokens': 7390, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 0.024102, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.024102, 'upstream_inference_prompt_cost': 0.021687, 'upstream_inference_completions_cost': 0.002415}}, 'model_provider': 'openai', 'model_name': 'anthropic/claude-son

## Generate routing KIs with a Kibana Workflow

Now we profile each index into a routing KI. The workflow loops over the three indices and, for each, chains four steps:

| Step | Type | What it does |
|------|------|--------------|
| `get_mapping` | `elasticsearch.request` | Read the mapping, including `_meta.description` and per-field descriptions. |
| `sample_docs` | `elasticsearch.search` | Pull a few real documents so the profile reflects actual value shapes. |
| `profile_index` | `ai.agent` | Generate a structured index profile as structured output. |
| `sink_index_ki` | `elasticsearch.request` | Write the profile into the AI Index as an `index_metadata_entry` KI, keyed on the index name so re-runs upsert in place. |

### Point the workflow at your GenAI connector (optional)

Leave blank to use Kibana's default GenAI connector, or paste a connector id to pin one.

In [12]:
LLM_CONNECTOR_ID = input(
    "Kibana GenAI connector id (blank = default connector): "
).strip()

Kibana GenAI connector id (blank = default connector):  


### Define the workflow

This is the same YAML you'd paste into the Workflows UI in the blog, templated so the sink writes to the AI Index you created above.

In [13]:
_WORKFLOW_YAML_TEMPLATE = """
version: '1'
name: beir-index-profile-ki
description: Profile an index into an index-selection Knowledge Indicator.
enabled: true
tags:
  - context-management
  - index-selection

triggers:
  - type: manual

consts:
  indices:
    - beir-fiqa
    - beir-nfcorpus
    - beir-scifact

steps:
  - name: loop_indices
    type: foreach
    foreach: '{{ consts.indices | json }}'
    iteration-on-failure:
      continue: true
    steps:
      - name: get_mapping
        type: elasticsearch.request
        with:
          method: GET
          path: '/{{ foreach.item }}/_mapping'

      - name: sample_docs
        type: elasticsearch.search
        with:
          index: '{{ foreach.item }}'
          size: 3
          query:
            match_all: {}

      - name: profile_index
        type: ai.agent
        connector-id: "__LLM_CONNECTOR_ID__"
        timeout: 120s
        with:
          message: >
            You are a data steward building an INDEX PROFILE for an enterprise
            data catalog. Downstream, an AI agent uses these profiles to decide
            WHICH Elasticsearch index to query for a given user question -- this
            is an index-SELECTION aid, not a place to answer the question itself.

            You are given (a) the index name, (b) its Elasticsearch mapping
            including human-written descriptions in `_meta.description` and each
            field's `meta.description`, and (c) a few sample documents. Produce a
            faithful, decision-useful profile. Rules:
            - Ground everything in the provided mapping + samples. Never invent
              fields, values, or purpose. If unknown, use an empty string/array.
            - Optimize for routing: make it obvious what kinds of questions this
              index can authoritatively answer, and what it canNOT.
            - Prefer concrete field names and real example values from the
              samples over vague phrasing.
            - For joins, surface shared keys (e.g. *_id fields) that link this
              index to sibling indices, since cross-index questions hinge on them.

            Index name: {{ foreach.item }}

            Elasticsearch mapping (JSON):
            {{ steps.get_mapping.output | json }}

            Sample documents (JSON):
            {{ steps.sample_docs.output.hits.hits | map: '_source' | json }}
          schema:
            type: object
            properties:
              display_name:
                type: string
                description: A concise human-readable name for what this index represents (<= 8 words).
              purpose:
                type: string
                description: 2-4 sentences describing what this index stores and its role. PRIMARY semantic surface for matching a question to this index.
              answers_questions:
                type: array
                items:
                  type: string
                description: 3-7 representative natural-language questions this index can authoritatively answer.
              does_not_contain:
                type: array
                items:
                  type: string
                description: 1-4 things a searcher might wrongly expect here but that live elsewhere, to prevent mis-routing.
              key_fields:
                type: array
                items:
                  type: string
                description: 3-10 of the most query-relevant fields as "field_name - what it is".
              when_to_use:
                type: string
                description: A single crisp routing heuristic - when should an agent pick THIS index? (<= 30 words).
              example_esql:
                type: string
                description: One realistic, runnable ES|QL query against this index answering one of answers_questions.
            required:
              - display_name
              - purpose
              - answers_questions
              - key_fields
              - when_to_use

      - name: sink_index_ki
        type: elasticsearch.request
        with:
          method: PUT
          path: '/__AI_INDEX__/_doc/{{ foreach.item | url_encode }}'
          body:
            '@timestamp': '{{ "now" | date: "%Y-%m-%dT%H:%M:%S.%LZ" }}'
            type: index_metadata_entry
            title: '{{ steps.profile_index.output.structured_output.display_name | default: foreach.item }}'
            tags:
              - index-profile
              - '{{ foreach.item }}'
            attributes:
              display_name: '{{ steps.profile_index.output.structured_output.display_name }}'
              purpose: '{{ steps.profile_index.output.structured_output.purpose }}'
              when_to_use: '{{ steps.profile_index.output.structured_output.when_to_use }}'
              answers_questions: '{{ steps.profile_index.output.structured_output.answers_questions | json }}'
              does_not_contain: '{{ steps.profile_index.output.structured_output.does_not_contain | json }}'
              key_fields: '{{ steps.profile_index.output.structured_output.key_fields | json }}'
              example_esql: '{{ steps.profile_index.output.structured_output.example_esql }}'
              source_index: '{{ foreach.item }}'
            content: >
              === SOURCE / PROVENANCE ===
              This is an INDEX PROFILE for routing/index-selection.
              Backing Elasticsearch index: {{ foreach.item }}
              Inspect it directly with ES|QL:
              FROM {{ foreach.item }} | LIMIT 10
              === WHAT THIS INDEX IS ===
              {{ steps.profile_index.output.structured_output.purpose }}
              Questions this index can answer: {{ steps.profile_index.output.structured_output.answers_questions | join: " | " }}
              When to use this index: {{ steps.profile_index.output.structured_output.when_to_use }}
              Example query:
              {{ steps.profile_index.output.structured_output.example_esql }}
            description: >
              Index profile: {{ steps.profile_index.output.structured_output.display_name }}.
              Does NOT contain: {{ steps.profile_index.output.structured_output.does_not_contain | join: "; " }}.
              Key fields: {{ steps.profile_index.output.structured_output.key_fields | join: "; " }}.
"""

# Pin a connector if one was supplied; otherwise drop the line so `ai.agent` uses
# Kibana's default GenAI connector (step-level config can't be templated at runtime).
_yaml = _WORKFLOW_YAML_TEMPLATE.replace("__AI_INDEX__", AI_INDEX)
if LLM_CONNECTOR_ID:
    WORKFLOW_YAML = _yaml.replace("__LLM_CONNECTOR_ID__", LLM_CONNECTOR_ID)
else:
    WORKFLOW_YAML = "\n".join(
        line for line in _yaml.splitlines() if "__LLM_CONNECTOR_ID__" not in line
    )

print(WORKFLOW_YAML)


version: '1'
name: beir-index-profile-ki
description: Profile an index into an index-selection Knowledge Indicator.
enabled: true
tags:
  - context-management
  - index-selection

triggers:
  - type: manual

consts:
  indices:
    - beir-fiqa
    - beir-nfcorpus
    - beir-scifact

steps:
  - name: loop_indices
    type: foreach
    foreach: '{{ consts.indices | json }}'
    iteration-on-failure:
      continue: true
    steps:
      - name: get_mapping
        type: elasticsearch.request
        with:
          method: GET
          path: '/{{ foreach.item }}/_mapping'

      - name: sample_docs
        type: elasticsearch.search
        with:
          index: '{{ foreach.item }}'
          size: 3
          query:
            match_all: {}

      - name: profile_index
        type: ai.agent
        timeout: 120s
        with:
          message: >
            You are a data steward building an INDEX PROFILE for an enterprise
            data catalog. Downstream, an AI agent uses thes

### Create and run the workflow

The `foreach` loop runs sequentially and each iteration makes an LLM call, so this takes a couple of minutes. For scale, use [workflow.executeAsync](https://www.elastic.co/docs/explore-analyze/workflows/steps/composition) or native parallel support.

In [14]:
WF_API_VERSION = "2023-10-31"
TERMINAL_STATES = {"completed", "failed", "cancelled", "timed_out", "skipped"}


def create_workflow(yaml_str):
    return kbn_request(
        "POST",
        "/api/workflows/workflow",
        body={"yaml": yaml_str},
        api_version=WF_API_VERSION,
    )


def run_workflow(workflow_id, inputs=None):
    return kbn_request(
        "POST",
        f"/api/workflows/workflow/{workflow_id}/run",
        body={"inputs": inputs or {}},
        api_version=WF_API_VERSION,
    )


def wait_for_execution(execution_id, timeout=1800, interval=5):
    deadline = time.time() + timeout
    start = time.time()
    last_status = None
    step_status = {}
    while time.time() < deadline:
        ex = kbn_request(
            "GET",
            f"/api/workflows/executions/{execution_id}?includeOutput=true",
            api_version=WF_API_VERSION,
        )
        if ex["status"] != last_status:
            last_status = ex["status"]
            print(f"  [{int(time.time() - start):>4}s] execution: {last_status}")
        for step in ex.get("stepExecutions", []):
            key = step.get("stepId")
            if step_status.get(key) != step.get("status"):
                step_status[key] = step.get("status")
                print(f"  [{int(time.time() - start):>4}s]   step {key}: {step.get('status')}")
        if ex["status"] in TERMINAL_STATES:
            return ex
        time.sleep(interval)
    raise TimeoutError(f"Execution {execution_id} did not finish within {timeout}s")

In [ ]:
wf = create_workflow(WORKFLOW_YAML)
workflow_id = wf["id"]
print("Created workflow:", workflow_id)

execution = run_workflow(workflow_id)
exec_id = execution["workflowExecutionId"]
print("Running execution:", exec_id, "-- this may take a few minutes...\n")

result = wait_for_execution(exec_id)
print("Status:", result["status"])
if result["status"] != "completed":
    if result.get("error"):
        print("Error:", result["error"].get("message", result["error"]))
    for step in result.get("stepExecutions", []):
        if step.get("status") == "failed":
            print(
                "Failed step:", step.get("stepId"), "->", json.dumps(step.get("error"))
            )

Created workflow: beir-index-profile-ki
Running execution: ddc966cc-1e1d-4f30-b633-e09c4baf6d81 -- this may take a few minutes...



### Inspect the results

Query the AI Index directly to confirm what the workflow wrote.

In [ ]:
resp = client.esql.query(
    query=f"""
        FROM {AI_INDEX}
        | WHERE type == "index_metadata_entry"
        | KEEP title, description, tags
        | LIMIT 10
    """,
    format="json",
)
cols = [c["name"] for c in resp["columns"]]
for row in resp["values"]:
    print(json.dumps(dict(zip(cols, row)), indent=2))

## Hand it to an agent — routing with KIs

Same harness, same question — but now the agent has the `query-ki` skill instead of `get_mapping`. It retrieves the routing profile first, picks the right index, then queries only that one. Compare the tool-call count with the baseline above.

In [ ]:
backend = FilesystemBackend(root_dir=".", virtual_mode=False)

ki_agent = create_deep_agent(
    model=agent_llm,
    tools=[esql_query],
    skills=["skills"],
    backend=backend,
    system_prompt=(
        "You are a research assistant with access to several Elasticsearch indices. "
        "You do NOT know which index is relevant for a given question. "
        "Before searching, always use the query-ki skill with type 'index_metadata_entry' "
        "to retrieve the routing profile for the right index, then query that index directly. "
        'Full-text search syntax: WHERE MATCH(field, "value") -- never use field MATCH "value". '
        "Ground your answer strictly in what the queries return and cite the KI you used for routing."
    ),
)

run_agent(ki_agent, QUESTION)

The KI answer is about equivalent, but it likely took fewer tool calls to get there — the agent no longer has to inspect every mapping and probe every index to figure out where to look. Multiply that saving across every query an agent makes and it adds up.

## Clean up

Remove the AI Index, the source indices, and the workflow.

In [ ]:
client.indices.delete(index=AI_INDEX, ignore_unavailable=True)
print("Deleted index:", AI_INDEX)

for ds in DATASETS:
    client.indices.delete(index=ds["index_name"], ignore_unavailable=True)
    print("Deleted index:", ds["index_name"])

try:
    kbn_request(
        "DELETE", f"/api/workflows/workflow/{workflow_id}", api_version=WF_API_VERSION
    )
    print("Deleted workflow:", workflow_id)
except Exception as e:
    print("Workflow delete skipped:", e)